In [10]:
import pandas as pd
import numpy as np

# Set a random seed so our "mock" numbers are exactly the same every time we run it
np.random.seed(42)

# Define how many total rows (historical + active employees) we want to mock up
num_employees = 250

# Generate a sequential list of unique Employee IDs (e.g., EMP001, EMP002...)
employee_ids = [f"EMP{str(i).zfill(3)}" for i in range(1, num_employees + 1)]

print(f"Generated {len(employee_ids)} unique Employee IDs. First 5: {employee_ids[:5]}")

Generated 250 unique Employee IDs. First 5: ['EMP001', 'EMP002', 'EMP003', 'EMP004', 'EMP005']


In [11]:
# Create an empty dictionary to hold our columns temporarily
data = {}

# 1. Assign Employee IDs
data['Employee_ID'] = employee_ids

# 2. Randomly assign Departments with specific probabilities
departments = ['Engineering', 'Data Science', 'Human Resources', 'Sales']
data['Department'] = np.random.choice(departments, size=num_employees, p=[0.4, 0.2, 0.1, 0.3])

# 3. Generate Monthly Salaries based on department averages
# Engineering/Data Science generally make more than HR/Sales in our mock company
base_salaries = {
    'Engineering': 9500,
    'Data Science': 10000,
    'Sales': 6000,
    'Human Resources': 5500
}
data['Monthly_Salary'] = [
    int(base_salaries[dept] + np.random.normal(0, 800)) for dept in data['Department']
]

# 4. Generate Job Satisfaction Score (1 to 5)
# Lower score means high role/skill misalignment
data['Job_Satisfaction'] = np.random.choice([1, 2, 3, 4, 5], size=num_employees, p=[0.1, 0.15, 0.25, 0.35, 0.15])

# 5. Convert our dictionary into a beautiful Pandas DataFrame table
df_workforce = pd.DataFrame(data)

# Display the first 10 rows of our table
df_workforce.head(10)

,Employee_ID,Department,Monthly_Salary,Job_Satisfaction
0,EMP001,Engineering,8822,5
1,EMP002,Sales,4788,5
2,EMP003,Sales,5642,2
3,EMP004,Data Science,10685,5
4,EMP005,Engineering,9671,3
5,EMP006,Engineering,8503,3
6,EMP007,Engineering,9638,3
7,EMP008,Sales,6308,5
8,EMP009,Human Resources,4792,3
9,EMP010,Sales,6122,3


In [12]:
# tells us exact data types and non-null values
print("--- 1. Structural Summary ---")
print(df_workforce.info())

# scorecard of missing values 1 & 0
print("\n--- 2. Missing Values Check ---")
print(df_workforce.isnull().sum())

# gives descriptive metrics of data
print("\n--- 3. Numerical Ranges (Health Check) ---")
print(df_workforce.describe())

#aggregates categorical data and # of ees in each department
print("\n--- 4. Value Distributions ---")
print(df_workforce['Department'].value_counts())

--- 1. Structural Summary ---
<class 'pandas.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Employee_ID       250 non-null    str  
 1   Department        250 non-null    str  
 2   Monthly_Salary    250 non-null    int64
 3   Job_Satisfaction  250 non-null    int64
dtypes: int64(2), str(2)
memory usage: 7.9 KB
None

--- 2. Missing Values Check ---
Employee_ID         0
Department          0
Monthly_Salary      0
Job_Satisfaction    0
dtype: int64

--- 3. Numerical Ranges (Health Check) ---
       Monthly_Salary  Job_Satisfaction
count      250.000000        250.000000
mean      8147.392000          3.256000
std       2102.514816          1.147051
min       3406.000000          1.000000
25%       6102.750000          2.000000
50%       8861.500000          3.000000
75%       9954.000000          4.000000
max      12582.000000          5.000000

--- 4. Value Dist

In [13]:
#SQLlite for Python 
import duckdb

# 1. Save our DataFrame as a CSV file to simulate our "Bronze Raw Data" landing
df_workforce.to_csv("bronze_workforce_raw.csv", index=False)
print("Raw data successfully saved to bronze_workforce_raw.csv!")

# 2. Connect to an in-memory DuckDB database instance
con = duckdb.connect(database=':memory:')

# 3. Use DuckDB to read the CSV directly using SQL and register it as a view
con.execute("CREATE VIEW raw_employees AS SELECT * FROM read_csv_auto('bronze_workforce_raw.csv')")

# 4. Run a test SQL query to make sure DuckDB sees our data
test_query = con.execute("SELECT * FROM raw_employees LIMIT 3").df()
print("\nDuckDB SQL Test Output:")
print(test_query)

Raw data successfully saved to bronze_workforce_raw.csv!

DuckDB SQL Test Output:
  Employee_ID   Department  Monthly_Salary  Job_Satisfaction
0      EMP001  Engineering            8822                 5
1      EMP002        Sales            4788                 5
2      EMP003        Sales            5642                 2


In [14]:
# 1. Create the Cleaned Dimension Table (Silver Layer)
# This extracts unique attributes about the employee
con.execute("""
    CREATE TABLE dim_employee AS
    SELECT DISTINCT 
        Employee_ID,
        Department
    FROM raw_employees
""")

# 2. Create the Cleaned Fact Table (Silver Layer)
# This extracts the quantitative metrics that change or get measured
con.execute("""
    CREATE TABLE fact_payroll_satisfaction AS
    SELECT 
        Employee_ID,
        Monthly_Salary,
        Job_Satisfaction
    FROM raw_employees
""")

print("Tables successfully created in DuckDB!")

# 3. Verify our new Star Schema by joining them back together using SQL
verification_query = """
    SELECT 
        f.Employee_ID,
        d.Department,
        f.Monthly_Salary,
        f.Job_Satisfaction
    FROM fact_payroll_satisfaction f
    JOIN dim_employee d ON f.Employee_ID = d.Employee_ID
    LIMIT 5
"""

df_verified = con.execute(verification_query).df()
print("\nVerified Star Schema Join Output:")
print(df_verified)

Tables successfully created in DuckDB!

Verified Star Schema Join Output:
  Employee_ID   Department  Monthly_Salary  Job_Satisfaction
0      EMP001  Engineering            8822                 5
1      EMP002        Sales            4788                 5
2      EMP003        Sales            5642                 2
3      EMP006  Engineering            8503                 3
4      EMP008        Sales            6308                 5


In [15]:
# 1. Define the Market Midpoints for our Compa-Ratio calculations
# In a full production system, this would be a separate dimension table
market_midpoints = {
    'Engineering': 9500,
    'Data Science': 10000,
    'Sales': 6500,
    'Human Resources': 5500
}

# 2. Extract our database layers back to Pandas to perform engineering logic
df_fact = con.execute("SELECT * FROM fact_payroll_satisfaction").df()
df_dim = con.execute("SELECT * FROM dim_employee").df()

# Merge them temporarily in Python to align metrics with departments
df_gold = pd.merge(df_fact, df_dim, on='Employee_ID')

# 3. ETL Transformation: Calculate Compa-Ratio
# Formula: Actual Salary / Market Midpoint
df_gold['Market_Midpoint'] = df_gold['Department'].map(market_midpoints)
df_gold['Compa_Ratio'] = round(df_gold['Monthly_Salary'] / df_gold['Market_Midpoint'], 2)

# 4. ETL Transformation: Calculate Retention Risk Score (Conditional Logic Engine)
# Logic: High risk if underpaid (Compa-Ratio < 0.85) AND dissatisfied (Satisfaction <= 2)
def calculate_risk(row):
    if row['Compa_Ratio'] < 0.85 and row['Job_Satisfaction'] <= 2:
        return 'High'
    elif row['Compa_Ratio'] < 0.95 or row['Job_Satisfaction'] <= 3:
        return 'Medium'
    else:
        return 'Low'

df_gold['Retention_Risk'] = df_gold.apply(calculate_risk, axis=1)

# 5. Production Quality Gate Check
# Verify no nulls exist and compa-ratios fall within realistic bounds (> 0)
null_count = df_gold['Compa_Ratio'].isnull().sum()
invalid_metrics = df_gold[df_gold['Compa_Ratio'] <= 0]

if null_count == 0 and len(invalid_metrics) == 0:
    print("✅ Quality Gate Passed: Data conforms to structural business rules.")
else:
    print("❌ Quality Gate Failed: Data anomalies detected!")

# Preview our production-ready Gold Layer
df_gold[['Employee_ID', 'Department', 'Monthly_Salary', 'Compa_Ratio', 'Retention_Risk']].head(10)

✅ Quality Gate Passed: Data conforms to structural business rules.


,Employee_ID,Department,Monthly_Salary,Compa_Ratio,Retention_Risk
0,EMP001,Engineering,8822,0.93,Medium
1,EMP002,Sales,4788,0.74,Medium
2,EMP003,Sales,5642,0.87,Medium
3,EMP004,Data Science,10685,1.07,Low
4,EMP005,Engineering,9671,1.02,Medium
5,EMP006,Engineering,8503,0.90,Medium
6,EMP007,Engineering,9638,1.01,Medium
7,EMP008,Sales,6308,0.97,Low
8,EMP009,Human Resources,4792,0.87,Medium
9,EMP010,Sales,6122,0.94,Medium
